# Stratified Analysis (v3)

Regenerates the stratification figures on the canonical **v3** 320-case set,
**pooled over the 5 variance runs** (so point estimates match the Table 1 headline
means).

- **Main text** — `fig_personality_comprehension`: post-education comprehension
  across the 5 patient personalities (hardest→easiest), one line per model. The
  per-model fan-out is the robustness signal an interactive benchmark surfaces.
- **Appendix** — `strat_<factor>_v3`: per-factor 2-panel heatmap (Comprehension |
  TCS), rows = models, shared column order.

Comprehension = `post_education_score`; safety = **Unsafe Claims**
(`unsafe_advice_rate + wrong_instruction_rate`, the two educator-attributable grid
cells). ROUGE-L / Similarity are omitted (not present in the per-run outputs).
Colors use the validated colorblind-safe categorical palette in a fixed model order
+ per-model markers, so identity never rests on color alone.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for _ttf in (PROJECT_ROOT / "fonts").glob("*.ttf"):
    fm.fontManager.addfont(str(_ttf))
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Fira Sans", "DejaVu Sans"]
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11

OUTPUT_ROOT = PROJECT_ROOT / "data/processed/output"
FIG_DIR = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
OUT_DIR = PROJECT_ROOT / "outputs"; OUT_DIR.mkdir(exist_ok=True)
RUN_TAGS = [f"run{i}" for i in range(1, 6)]

# (display name, output dir) in FIXED order = palette-slot order (never cycled)
MODELS = [("GPT-5.5", "gpt-5.5"), ("GPT-5.4-mini", "gpt-5.4-mini"),
          ("GPT-5.4-nano", "gpt-5.4-nano"), ("Qwen3-32B", "Qwen3-32B"),
          ("Qwen3-4B", "Qwen3-4B"), ("MedGemma-27B", "medgemma-27b-text-it"),
          ("Llama-3.3-70B", "Llama-3.3-70B-Instruct"), ("MedGemma-4B", "medgemma-4b-it")]
DISP = [d for d, _ in MODELS]
# unified manuscript palette, by model (GPT purples, Qwen greens, MedGemma warm, Llama blue)
PALETTE = ["#3D2168", "#6E549A", "#B7A3CC", "#326859",
           "#7BB4A0", "#A15340", "#3B5F91", "#E1B39D"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
COLOR = dict(zip(DISP, PALETTE)); MARK = dict(zip(DISP, MARKERS))

PERS_LABEL = {"distrustful": "Distrustful", "anxious": "Anxious", "minimiser": "Minimizer",
              "neutral": "Neutral", "high_conscientiousness": "Conscientious"}
FACTOR_TITLE = {"personality": "Personality", "health_literacy": "Health literacy",
                "education_level": "Education level",
                "past_medical_history_recall": "History recall", "icd_chapter": "ICD chapter"}
ICD_SHORT = {
    "Congenital Anomalies": "Congenital",
    "Diseases of the Blood and Blood-Forming Organs": "Blood",
    "Diseases of the Circulatory System": "Circulatory",
    "Diseases of the Digestive System": "Digestive",
    "Diseases of the Genitourinary System": "Genitourinary",
    "Diseases of the Musculoskeletal System and Connective Tissue": "Musculoskeletal",
    "Diseases of the Nervous System": "Nervous",
    "Diseases of the Respiratory System": "Respiratory",
    "Diseases of the Skin and Subcutaneous Tissue": "Skin",
    "Endocrine, Nutritional and Metabolic Diseases": "Endocrine",
    "Factors Influencing Health Status": "Factors",
    "Infectious and Parasitic Diseases": "Infectious",
    "Injury and Poisoning": "Injury",
    "Neoplasms": "Neoplasms",
    "Pregnancy, Childbirth and the Puerperium": "Pregnancy",
    "Symptoms, Signs and Ill-Defined Conditions": "Symptoms",
}
ICD_ABBR = {
    "Neoplasms": "NEO", "Diseases of the Circulatory System": "CIR",
    "Diseases of the Musculoskeletal System and Connective Tissue": "MSK",
    "Diseases of the Genitourinary System": "GEN", "Diseases of the Digestive System": "DIG",
    "Diseases of the Respiratory System": "RES", "Diseases of the Blood and Blood-Forming Organs": "BLD",
    "Diseases of the Skin and Subcutaneous Tissue": "SKN", "Diseases of the Nervous System": "NER",
    "Infectious and Parasitic Diseases": "IPD", "Injury and Poisoning": "INJ",
    "Endocrine, Nutritional and Metabolic Diseases": "END", "Factors Influencing Health Status": "FHS",
    "Symptoms, Signs and Ill-Defined Conditions": "SSI",
    "Pregnancy, Childbirth and the Puerperium": "PCP", "Congenital Anomalies": "CON",
}
FACTORS = ["personality", "health_literacy", "education_level",
           "past_medical_history_recall", "icd_chapter"]
# factors with a natural (ordinal) level order -> use it instead of score order
FIXED_ORDER = {"education_level": ["elementary", "high_school", "college"]}

curated = pd.read_json(PROJECT_ROOT / "data/processed/discharge_bench_curated_v3.jsonl", lines=True)
CURATED_IDS = set(curated["note_id"])
meta = curated[["note_id", "icd_chapter"]].copy()
for ax in ["personality", "health_literacy", "education_level", "past_medical_history_recall"]:
    meta[ax] = curated["profile"].apply(lambda p: p.get(ax))
print(f"v3 cases: {len(CURATED_IDS)}")

In [ ]:
# --- pool the 5 variance runs, per case, into a long dataframe -----------
def _cq(key):
    return lambda r: r["conversation_quality"][key]["score"]

METRIC_SPEC = {
    # full-word keys: "NA" round-trips as NaN through pandas read_csv
    "Naturalness": ("conversation_quality_score.jsonl", _cq("naturalness")),
    "Responsiveness": ("conversation_quality_score.jsonl", _cq("responsiveness")),
    "Clarity": ("conversation_quality_score.jsonl", _cq("clarity")),
    "ClinicalRelevance": ("conversation_quality_score.jsonl", _cq("clinical_relevance")),
    "Readability": ("conversation_quality_score.jsonl", lambda r: r["readability"]["score"]),
    "TopicChecklist": ("topic_checklist_score.jsonl", lambda r: r["tcs_score"]),
    # Comprehension (SCS, the reported score): persona-ablated patient, all six questions, items whose
    # reference is "not applicable" dropped (scripts/scoring/score_comprehension_on.py).
    "Comprehension": ("comprehension_score_orig_na.jsonl", lambda r: r["post_education_score"]),
    # variants for the persona-leak comparison: strict persona, all six questions; standard persona
    "Comprehension_strict6": ("comprehension_score_stripped.jsonl", lambda r: r["post_education_score"]),
    "Comprehension_std": ("comprehension_score.jsonl", lambda r: r["post_education_score"]),
    "UnsafeClaims": ("safety_commission_score.jsonl",
                     lambda r: r["unsafe_advice_rate"] + r["wrong_instruction_rate"]),
}

def pooled(model_dir, fname, extractor):
    acc = {}
    for rt in RUN_TAGS:
        p = OUTPUT_ROOT / f"{model_dir}__{rt}" / fname
        if not p.exists():
            continue
        for line in open(p):
            if not line.strip():
                continue
            row = json.loads(line)
            if row["note_id"] not in CURATED_IDS:
                continue
            try:
                v = extractor(row)
            except (KeyError, TypeError):
                v = None
            if v is not None:
                acc.setdefault(row["note_id"], []).append(float(v))
    return {k: float(np.mean(v)) for k, v in acc.items()}

records = []
for disp, mdir in MODELS:
    for metric, (fname, ex) in METRIC_SPEC.items():
        for nid, val in pooled(mdir, fname, ex).items():
            records.append((disp, nid, metric, val))
long = (pd.DataFrame(records, columns=["Model", "note_id", "metric", "value"])
        .merge(meta, on="note_id", how="left"))
long.to_csv(OUT_DIR / "stratified_long_v3.csv", index=False)
print(f"{len(long)} rows | metrics: {sorted(long.metric.unique())}")
long.head()

In [ ]:
# --- helpers + factor-effect summary (which factors move the metrics) ---
def level_order(metric, factor):
    gm = long[long.metric == metric].groupby(factor)["value"].mean()
    return list(gm.sort_values().index)

def model_order(metric):
    gm = long[long.metric == metric].groupby("Model")["value"].mean()
    return list(gm.sort_values(ascending=False).index)   # best first

def level_table(metric, factor):
    return long[long.metric == metric].groupby(["Model", factor])["value"].mean().unstack()

rows = []
for metric in ["Comprehension", "TopicChecklist", "UnsafeClaims", "Readability", "Naturalness"]:
    for factor in FACTORS:
        gm = long[long.metric == metric].groupby(factor)["value"].mean()
        rows.append({"metric": metric, "factor": factor, "spread": gm.max() - gm.min()})
effect = pd.DataFrame(rows).pivot(index="factor", columns="metric", values="spread")
print("Per-factor spread (pooled across models) — larger = factor moves the metric more:")
effect.round(4)

## Main-text figure — personality × comprehension

In [ ]:
factor = "personality"
order = level_order("Comprehension", factor)          # hardest -> easiest
tbl = level_table("Comprehension", factor)
x = np.arange(len(order))

fig, ax = plt.subplots(figsize=(7.6, 4.8))
all_y = []
for disp in DISP:
    ys = [tbl.loc[disp, L] for L in order]; all_y += ys
    ax.plot(x, ys, marker=MARK[disp], color=COLOR[disp], lw=2, ms=7, label=disp, zorder=3)
lo, hi = min(all_y), max(all_y); pad = 0.04 * (hi - lo)
ax.set_xlim(-0.15, len(order) - 1 + 0.15); ax.set_ylim(lo - pad, hi + pad)
ax.set_xticks(x); ax.set_xticklabels([PERS_LABEL[L] for L in order])
ax.set_ylabel("SCS"); ax.set_xlabel("Patient personality")
ax.grid(axis="y", color="0.85", lw=0.6, zorder=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(ncol=4, fontsize=10, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / f"fig_personality_comprehension.{ext}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# TCS twin of Figure 7: same style and SAME x order (Comprehension hardest -> easiest)
factor = "personality"
order = level_order("Comprehension", factor)
tbl = level_table("TopicChecklist", factor)
x = np.arange(len(order))

fig, ax = plt.subplots(figsize=(7.6, 4.8))
all_y = []
for disp in DISP:
    ys = [tbl.loc[disp, L] for L in order]; all_y += ys
    ax.plot(x, ys, marker=MARK[disp], color=COLOR[disp], lw=2, ms=7, label=disp, zorder=3)
lo, hi = min(all_y), max(all_y); pad = 0.04 * (hi - lo)
ax.set_xlim(-0.15, len(order) - 1 + 0.15); ax.set_ylim(lo - pad, hi + pad)
ax.set_xticks(x); ax.set_xticklabels([PERS_LABEL[L] for L in order])
ax.set_ylabel("Topic Checklist Score"); ax.set_xlabel("Patient personality")
ax.grid(axis="y", color="0.85", lw=0.6, zorder=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(ncol=4, fontsize=10, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / f"fig_personality_tcs.{ext}", dpi=300, bbox_inches="tight")
plt.show()

### Alternative: per-model dot plot (personality is nominal, so no connecting lines)

In [ ]:
# rows = models (best overall at top); markers = personalities; bar = worst..best persona
import matplotlib as mpl
factor = "personality"
order = level_order("Comprehension", factor)              # hardest -> easiest
tbl = level_table("Comprehension", factor)
rows = model_order("Comprehension")[::-1]                  # best at top after invert
P_COL = dict(zip(order, mpl.colormaps["cividis"](np.linspace(0.0, 0.9, len(order)))))
P_MARK = dict(zip(order, ["o", "s", "D", "^", "v"]))

fig, ax = plt.subplots(figsize=(7.6, 4.8))
for i, disp in enumerate(rows):
    v = tbl.loc[disp, order]
    ax.hlines(i, v.min(), v.max(), color="0.75", lw=5, zorder=1, capstyle="round")
    for L in order:
        ax.scatter(v[L], i, s=58, marker=P_MARK[L], color=P_COL[L],
                   edgecolor="white", lw=0.6, zorder=3, label=PERS_LABEL[L] if i == 0 else None)
    ax.text(v.max() + 0.008, i, f"Δ {v.max() - v.min():.2f}", va="center", fontsize=10, color="0.35")
ax.set_yticks(range(len(rows)))
ax.set_yticklabels(rows)
ax.set_xlabel("SCS")
ax.set_xlim(right=ax.get_xlim()[1] + 0.04)
ax.grid(True, color="0.88", lw=0.6, zorder=0); ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(title="Patient personality", ncol=5, fontsize=10, title_fontsize=10, frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, -0.14), handletextpad=0.2, columnspacing=1.0)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / f"fig_personality_comprehension_dot.{ext}", dpi=300, bbox_inches="tight")
plt.show()

## Appendix figures — per-factor stratification heatmaps (Comprehension | TCS)

In [ ]:
def short_levels(factor, levels):
    if factor == "personality":
        return [PERS_LABEL[L] for L in levels]
    if factor == "icd_chapter":
        return [ICD_SHORT.get(str(L), str(L)) for L in levels]
    return [str(L).replace("_", " ").title() for L in levels]

def heatmap_factor(factor, panels=(("Comprehension", "SCS"), ("TopicChecklist", "TCS")),
                   stem=None, levels=None):
    rows = model_order("Comprehension")
    # fixed ordinal order where defined (e.g. education level), else by comprehension
    levels = levels or FIXED_ORDER.get(factor) or level_order("Comprehension", factor)
    wide = factor == "icd_chapter"
    fig, axes = plt.subplots(1, 2, figsize=(13 if wide else 8.4, 4.2))
    for ax, (metric, mlabel) in zip(axes, panels):
        tbl = level_table(metric, factor)
        M = np.array([[tbl.loc[m, L] for L in levels] for m in rows])
        im = ax.imshow(M, cmap="cividis", aspect="auto")
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels(short_levels(factor, levels), rotation=45, ha="right", fontsize=10)
        ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=10)
        ax.set_title(mlabel, fontsize=13, fontweight="bold")
        thr = np.nanmin(M) + 0.55 * (np.nanmax(M) - np.nanmin(M))
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                            color="white" if M[i, j] < thr else "black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()                       # no suptitle: the paper caption carries it
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{stem or f'strat_{factor}_v3'}.{ext}", dpi=300, bbox_inches="tight")
    plt.show()

for f in FACTORS:
    heatmap_factor(f)

# appendix: conversation-quality axes by personality (same column order as strat_personality_v3)
heatmap_factor("personality",
               panels=(("Naturalness", "Naturalness (1-5)"), ("Readability", "Readability (FK grade)")),
               stem="strat_personality_cq_v3")

## Appendix — ICD-chapter difficulty pooled across all models

In [ ]:
_RNG = np.random.default_rng(0)

def _boot_ci(x, n_boot=5000, ci=95):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    boot = _RNG.choice(x, size=(n_boot, x.size), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(x.mean()), float(lo), float(hi)

AGG_METRICS = ["Comprehension", "TopicChecklist", "UnsafeClaims", "Readability",
               "Naturalness", "Responsiveness", "Clarity", "ClinicalRelevance"]
AGG_LABELS = {"Comprehension": "Comprehension", "TopicChecklist": "Topic Checklist",
              "UnsafeClaims": "Unsafe Claims", "Readability": "Readability (grade)",
              "Naturalness": "Naturalness", "Responsiveness": "Responsiveness",
              "Clarity": "Clarity", "ClinicalRelevance": "Clinical Relevance"}

axis = "icd_chapter"
sub = long[long[axis].notna()]
levels = level_order("Comprehension", axis)          # hardest comprehension first
abbrs = [ICD_ABBR[lv] for lv in levels]
xpos = np.arange(len(levels))

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, metric in zip(axes.flat, AGG_METRICS):
    m_sub = sub[sub.metric == metric]
    means, los, his = [], [], []
    for lv in levels:
        vals = m_sub.loc[m_sub[axis] == lv, "value"].to_numpy(float)
        m, lo, hi = _boot_ci(vals)
        means.append(m); los.append(lo); his.append(hi)
    means = np.array(means); los = np.array(los); his = np.array(his)
    err = np.nan_to_num(np.vstack([means - los, his - means]), nan=0.0)
    ax.bar(xpos, means, 0.75, yerr=err, capsize=2, color="#2a78d6",
           edgecolor="0.3", linewidth=0.4, error_kw={"ecolor": "0.3", "lw": 0.8})
    ax.axhline(np.nanmean(m_sub["value"].to_numpy(float)), color="0.4", lw=0.9, ls="--")
    ax.set_title(AGG_LABELS[metric], fontsize=12, fontweight="bold")
    ax.set_xticks(xpos); ax.set_xticklabels(abbrs, rotation=45, ha="right", fontsize=9)
fig.suptitle("ICD-chapter difficulty, pooled across all models "
             "(v3; 95% bootstrap CI, dashed = overall mean)", fontsize=13, y=1.01)
legend = "     ".join(f"{ICD_ABBR[lv]} = {ICD_SHORT.get(lv, lv)}" for lv in levels)
fig.text(0.5, -0.03, legend, ha="center", fontsize=7.5)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / f"strat_icd_aggregated_v3.{ext}", dpi=300, bbox_inches="tight")
plt.show()

## Comprehension vs. TCS divergence by ICD chapter

In [ ]:
# |Comprehension - TCS| per model x ICD chapter: where topic coverage (TCS) and
# patient understanding (Comprehension) diverge. A large gap = the educator covered
# the checklist topics but the patient did not end up understanding them (or vice versa).
# uses the STRICT (persona-ablated) comprehension = the default reported metric.
# Under strict, the big persona-leak gaps close; the residual gaps (e.g. MSK) are
# genuine coverage-without-understanding (educator covers it, patient doesn't absorb).
levels = level_order("Comprehension", "icd_chapter")
rows = model_order("Comprehension")
comp = level_table("Comprehension", "icd_chapter").loc[rows, levels]
tcs = level_table("TopicChecklist", "icd_chapter").loc[rows, levels]
gap = (comp - tcs).abs()
M = gap.to_numpy()

fig, ax = plt.subplots(figsize=(13, 4.6))
im = ax.imshow(M, cmap="Reds", aspect="auto")
ax.set_xticks(range(len(levels)))
ax.set_xticklabels([ICD_SHORT.get(l, l) for l in levels], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=8)
thr = np.nanmin(M) + 0.55 * (np.nanmax(M) - np.nanmin(M))
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if M[i, j] > thr else "black")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="|Comprehension - TCS|")
ax.set_title("Comprehension vs. TCS gap by model and ICD chapter (v3)", fontsize=11)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / f"strat_icd_comp_tcs_gap_v3.{ext}", dpi=300, bbox_inches="tight")
plt.show()